## Inferencia sobre validacion - metricas CV (MTL)

Calcula las metricas de los 5 folds de cada modelo multitarea sobre su conjunto de **validacion** retenido (reconstruido con el mismo `StratifiedKFold(random_state=42)` del entrenamiento, estratificado por la etiqueta de diagnostico, como en la CV MTL). Carga los `best_weights` guardados y predice las 3 tareas; no reentrena.

Notebook autocontenido. Solo hay que poner las carpetas de experimento reales en `MODELS`.

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, roc_auc_score,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"   

train_csv  = "/home/marc/MARIADELMAR_EXPERIMENTS/onehot_files/train_onehot.csv"
val_csv    = "/home/marc/MARIADELMAR_EXPERIMENTS/onehot_files/val_onehot.csv"
images_dir = "/home/marc/MARIADELMAR_EXPERIMENTS/dataverse_files/images"
sym_csv    = "/home/marc/MARIADELMAR_EXPERIMENTS/ham10000_shape_symmetry_ALL.csv"

N_FOLDS         = 5
NUM_CLASSES     = 7
NUM_SYM_CLASSES = 3
NUM_MAL_CLASSES = 2
class_cols  = ["dx_akiec", "dx_bcc", "dx_bkl", "dx_df", "dx_mel", "dx_nv", "dx_vasc"]
class_names = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
sym_names   = ["2_ejes", "1_eje", "asimetrica"]
mal_names   = ["benigno", "maligno"]
MALIGN_CLASSES = ["dx_mel", "dx_bcc", "dx_akiec"]

# Conjunto train+val con las 3 etiquetas, mismo split que en entrenamiento
df_trainval = pd.concat([pd.read_csv(train_csv), pd.read_csv(val_csv)], ignore_index=True)
df_sym = pd.read_csv(sym_csv).rename(columns={"image": "image_id"})[["image_id", "shape_symmetry"]]
df_trainval = df_trainval.merge(df_sym, on="image_id", how="inner").reset_index(drop=True)
df_trainval["malignant"] = df_trainval[MALIGN_CLASSES].sum(axis=1).astype(int)
df_trainval["filepath"] = df_trainval["image_id"].apply(lambda x: os.path.join(images_dir, f"{x}.jpg"))

y_trainval_int = np.argmax(df_trainval[class_cols].values.astype("float32"), axis=1)  # estratifica por diagnostico

skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(skf.split(np.zeros(len(df_trainval)), y_trainval_int))


def make_ds(filepaths, img_size, preprocess_fn, batch=32):
    def _load(fp):
        img = tf.io.read_file(fp)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32)
        return preprocess_fn(img)
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(_load, num_parallel_calls=15)
    return ds.batch(batch).prefetch(50)


def metrics_task(y_true_int, proba, task, n_cls):
    pred = np.argmax(proba, axis=1)
    m = {
        f"{task}_acc":             float((y_true_int == pred).mean()),
        f"{task}_balanced_acc":    float(balanced_accuracy_score(y_true_int, pred)),
        f"{task}_precision_macro": float(precision_score(y_true_int, pred, average="macro", zero_division=0)),
        f"{task}_recall_macro":    float(recall_score(y_true_int, pred, average="macro", zero_division=0)),
        f"{task}_f1_macro":        float(f1_score(y_true_int, pred, average="macro", zero_division=0)),
        f"{task}_f1_weighted":     float(f1_score(y_true_int, pred, average="weighted", zero_division=0)),
        f"{task}_kappa":           float(cohen_kappa_score(y_true_int, pred)),
    }
    try:
        m[f"{task}_auc_macro"] = float(roc_auc_score(np.eye(n_cls)[y_true_int], proba,
                                                     multi_class="ovr", average="macro"))
    except Exception:
        m[f"{task}_auc_macro"] = float("nan")
    return m

2026-06-23 12:08:57.024005: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1


In [2]:
preprocess_vgg16       = lambda img: tf.keras.applications.vgg16.preprocess_input(img)
preprocess_resnet50    = lambda img: tf.keras.applications.resnet.preprocess_input(img)
preprocess_mobilenetv2 = lambda img: tf.keras.applications.mobilenet_v2.preprocess_input(img)
preprocess_densenet169 = lambda img: tf.keras.applications.densenet.preprocess_input(img)
preprocess_unet        = lambda img: img / 255.0

In [3]:
def _mtl_heads(base, img_size, arch):
    base.trainable = False
    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    shared = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    shared = tf.keras.layers.Dropout(0.3, name="shared_dropout")(shared)
    out_d = tf.keras.layers.Dense(NUM_CLASSES,     activation="softmax", name="head_disease")(shared)
    out_s = tf.keras.layers.Dense(NUM_SYM_CLASSES, activation="softmax", name="head_sym")(shared)
    out_m = tf.keras.layers.Dense(NUM_MAL_CLASSES, activation="softmax", name="head_mal")(shared)
    return tf.keras.Model(inputs, [out_d, out_s, out_m], name=f"MTL_{arch}")

def build_vgg16(img_size):
    return _mtl_heads(tf.keras.applications.VGG16(include_top=False, weights="imagenet",
                      input_shape=(img_size[0], img_size[1], 3)), img_size, "VGG16")
def build_resnet50(img_size):
    return _mtl_heads(tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
                      input_shape=(img_size[0], img_size[1], 3)), img_size, "ResNet50")
def build_mobilenetv2(img_size):
    return _mtl_heads(tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                      input_shape=(img_size[0], img_size[1], 3)), img_size, "MobileNetV2")
def build_densenet169(img_size):
    return _mtl_heads(tf.keras.applications.DenseNet169(include_top=False, weights="imagenet",
                      input_shape=(img_size[0], img_size[1], 3)), img_size, "DenseNet169")

def build_unet(img_size):
    # U-Net autoencoder MTL: 4 salidas [disease, sym, mal, reconstruccion]. En el bucle se toman [0],[1],[2].
    def enc(inp, f):
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(inp)
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        return x, tf.keras.layers.MaxPool2D(2,2)(x)
    def dec(inp, skip, f):
        u = tf.keras.layers.UpSampling2D(2)(inp)
        x = tf.keras.layers.Conv2D(f,2,activation="relu",padding="same",kernel_initializer="he_normal")(u)
        x = tf.keras.layers.concatenate([skip, x], axis=3)
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        return x
    inputs = tf.keras.layers.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    c1,p1 = enc(inputs,64); c2,p2 = enc(p1,128); c3,p3 = enc(p2,256); c4,p4 = enc(p3,512)
    b1 = tf.keras.layers.Conv2D(1024,3,activation="relu",padding="same",kernel_initializer="he_normal")(p4)
    b1 = tf.keras.layers.BatchNormalization()(b1)
    b1 = tf.keras.layers.Conv2D(1024,3,activation="relu",padding="same",kernel_initializer="he_normal",name="bottleneck_conv")(b1)
    b1 = tf.keras.layers.BatchNormalization(name="bottleneck_bn")(b1)
    e1 = dec(b1,c4,512); e2 = dec(e1,c3,256); e3 = dec(e2,c2,128); e4 = dec(e3,c1,64)
    fr = tf.keras.layers.Conv2D(64,3,activation="relu",padding="same",kernel_initializer="he_normal")(e4)
    fr = tf.keras.layers.BatchNormalization()(fr)
    recon = tf.keras.layers.Conv2D(3,1,activation="relu",padding="same",name="reconstruction_output")(fr)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(b1)
    shared = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    shared = tf.keras.layers.Dropout(0.3, name="shared_dropout")(shared)
    out_d = tf.keras.layers.Dense(NUM_CLASSES,     activation="softmax", name="head_disease")(shared)
    out_s = tf.keras.layers.Dense(NUM_SYM_CLASSES, activation="softmax", name="head_sym")(shared)
    out_m = tf.keras.layers.Dense(NUM_MAL_CLASSES, activation="softmax", name="head_mal")(shared)
    return tf.keras.Model(inputs, [out_d, out_s, out_m, recon], name="MTL_UNet_autoencoder")

In [4]:
BASE_MTL = Path("/home/marc/MARIADELMAR_EXPERIMENTS/MTL_experimentos")

# Pon las carpetas de experimento REALES de la CV MTL de cada modelo.
MODELS = [
    {"name":"VGG16",       "img_size":(224,224), "preprocess":preprocess_vgg16,
     "build_fn":build_vgg16,       "exp_dir":BASE_MTL/"VGG16_MTL"/"exp_2026-04-24_23-58_CUT7_5fold_v2"},
    {"name":"ResNet50",    "img_size":(224,224), "preprocess":preprocess_resnet50,
     "build_fn":build_resnet50,    "exp_dir":BASE_MTL/"ResNet50_MTL"/"exp_2026-04-25_09-34_CUT39_5fold_v2"},
    {"name":"MobileNetV2", "img_size":(224,224), "preprocess":preprocess_mobilenetv2,
     "build_fn":build_mobilenetv2, "exp_dir":BASE_MTL/"MobileNetV2_MTL"/"exp_2026-04-26_22-39_CUT0_5fold_v2"},
    {"name":"DenseNet169", "img_size":(224,224), "preprocess":preprocess_densenet169,
     "build_fn":build_densenet169, "exp_dir":BASE_MTL/"DenseNet_MTL"/"exp_2026-04-24_13-04_CUT0_5fold_v2"},
    {"name":"UNet",        "img_size":(256,256), "preprocess":preprocess_unet,
     "build_fn":build_unet,        "exp_dir":BASE_MTL/"UNet_MTL"/"exp_2026-06-21_23-14_5fold"},
]

print("Verificando rutas de experimentos:")
for m in MODELS:
    print(f"  [{'OK' if m['exp_dir'].exists() else 'FALTA'}] {m['name']:<14} {m['exp_dir']}")

Verificando rutas de experimentos:
  [OK] VGG16          /home/marc/MARIADELMAR_EXPERIMENTS/MTL_experimentos/VGG16_MTL/exp_2026-04-24_23-58_CUT7_5fold_v2
  [OK] ResNet50       /home/marc/MARIADELMAR_EXPERIMENTS/MTL_experimentos/ResNet50_MTL/exp_2026-04-25_09-34_CUT39_5fold_v2
  [OK] MobileNetV2    /home/marc/MARIADELMAR_EXPERIMENTS/MTL_experimentos/MobileNetV2_MTL/exp_2026-04-26_22-39_CUT0_5fold_v2
  [OK] DenseNet169    /home/marc/MARIADELMAR_EXPERIMENTS/MTL_experimentos/DenseNet_MTL/exp_2026-04-24_13-04_CUT0_5fold_v2
  [OK] UNet           /home/marc/MARIADELMAR_EXPERIMENTS/MTL_experimentos/UNet_MTL/exp_2026-06-21_23-14_5fold


In [7]:
for cfg in MODELS:
    name    = cfg["name"]
    exp_dir = Path(cfg["exp_dir"])
    print(name)

    rows = []
    for fold_idx, (train_idx, val_idx) in enumerate(splits, 1):
        fold_dir = exp_dir / f"fold_{fold_idx}"
        if not (fold_dir / "best_weights.index").exists():
            print(f"  fold {fold_idx}: sin pesos, se salta")
            continue

        df_vl = df_trainval.iloc[val_idx].reset_index(drop=True)
        y_d = np.argmax(df_vl[class_cols].values.astype("float32"), axis=1)
        y_s = df_vl["shape_symmetry"].values.astype(int)
        y_m = df_vl["malignant"].values.astype(int)
        val_ds = make_ds(df_vl["filepath"].values, cfg["img_size"], cfg["preprocess"])

        tf.keras.backend.clear_session()
        built = cfg["build_fn"](cfg["img_size"])
        model = built[0] if isinstance(built, tuple) else built
        for xb in val_ds.take(1):
            _ = model(xb[:1], training=False)
        model.load_weights(str(fold_dir / "best_weights")).expect_partial()

        preds = model.predict(val_ds, verbose=0)   
        proba_d, proba_s, proba_m = preds[0], preds[1], preds[2]

        m = {"fold": fold_idx}
        m.update(metrics_task(y_d, proba_d, "disease", NUM_CLASSES))
        
        try:
            m["auc_melanoma"] = float(roc_auc_score((y_d == 4).astype(int), proba_d[:, 4]))
        except Exception:
            m["auc_melanoma"] = float("nan")
        m.update(metrics_task(y_s, proba_s, "sym",     NUM_SYM_CLASSES))
        m.update(metrics_task(y_m, proba_m, "mal",     NUM_MAL_CLASSES))
        rows.append(m)
        print(f"  fold {fold_idx}: f1_d={m['disease_f1_macro']:.4f}  "
              f"f1_s={m['sym_f1_macro']:.4f}  f1_m={m['mal_f1_macro']:.4f}")

    df_val = pd.DataFrame(rows)
    df_val.to_csv(exp_dir / "all_folds_VAL_metrics.csv", index=False)
    num  = [c for c in df_val.columns if c != "fold"]
    summ = df_val[num].agg(["mean", "median", "std"]).T
    summ.columns = ["mean", "median", "std"]
    summ.to_csv(exp_dir / "summary_VAL_mean_median_std.csv")
    print(f"  rendimiento esperado (validacion) - {name}:")
    print(summ.loc[["disease_f1_macro", "sym_f1_macro", "mal_f1_macro"], ["mean", "std"]].round(4))
    print(f"  guardado en: {exp_dir}")
    print()

VGG16
  fold 1: f1_d=0.6708  f1_s=0.5007  f1_m=0.8190
  fold 2: f1_d=0.6546  f1_s=0.5362  f1_m=0.8119
  fold 3: f1_d=0.7173  f1_s=0.4944  f1_m=0.8238
  fold 4: f1_d=0.6845  f1_s=0.5322  f1_m=0.8133
  fold 5: f1_d=0.7075  f1_s=0.4202  f1_m=0.8305
  rendimiento esperado (validacion) — VGG16:
                    mean     std
disease_f1_macro  0.6869  0.0258
sym_f1_macro      0.4967  0.0466
mal_f1_macro      0.8197  0.0077
  guardado en: /home/marc/MARIADELMAR_EXPERIMENTS/MTL_experimentos/VGG16_MTL/exp_2026-04-24_23-58_CUT7_5fold_v2

ResNet50
  fold 1: f1_d=0.6930  f1_s=0.5546  f1_m=0.8287
  fold 2: f1_d=0.7215  f1_s=0.5511  f1_m=0.8346
  fold 3: f1_d=0.6996  f1_s=0.4869  f1_m=0.8127
  fold 4: f1_d=0.6278  f1_s=0.5329  f1_m=0.8123
  fold 5: f1_d=0.7422  f1_s=0.5246  f1_m=0.8383
  rendimiento esperado (validacion) — ResNet50:
                    mean     std
disease_f1_macro  0.6968  0.0432
sym_f1_macro      0.5300  0.0271
mal_f1_macro      0.8253  0.0122
  guardado en: /home/marc/MARIADELM

In [8]:
# Comparacion val vs test (F1 macro de diagnostico) por modelo
for cfg in MODELS:
    exp_dir = Path(cfg["exp_dir"])
    val_csv_f  = exp_dir / "all_folds_VAL_metrics.csv"
    test_csv_f = exp_dir / "all_folds_metrics.csv"
    if not (val_csv_f.exists() and test_csv_f.exists()):
        print(f"{cfg['name']}: falta algun CSV, se salta"); continue
    f1_val  = pd.read_csv(val_csv_f)["disease_f1_macro"].mean()
    f1_test = pd.read_csv(test_csv_f)["disease_f1_macro"].mean()
    print(f"{cfg['name']:<14} disease  f1_val={f1_val:.4f}  f1_test={f1_test:.4f}  brecha={f1_val - f1_test:+.4f}")

VGG16          disease  f1_val=0.6869  f1_test=0.6775  brecha=+0.0094
ResNet50       disease  f1_val=0.6968  f1_test=0.6883  brecha=+0.0085
MobileNetV2    disease  f1_val=0.6805  f1_test=0.6740  brecha=+0.0065
DenseNet169    disease  f1_val=0.7195  f1_test=0.6859  brecha=+0.0336
UNet           disease  f1_val=0.4188  f1_test=0.4138  brecha=+0.0050
